# [LAB12] 딥러닝 > 신경망의 이해 > 09. 이항분류

진돗개, 닥스훈트 분류 데이터

## 📘 #01. 준비작업

### 📝 [1] 패키지 설치 (Colab으로 실행하는 경우)

### 📝 [2] 패키지 가져오기

In [1]:
!pip install --upgrade hossam keras-tuner

   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 36.6 MB/s  0:00:00
  Attempting uninstall: hossam
    Found existing installation: hossam 0.5.3
    Uninstalling hossam-0.5.3:
      Successfully uninstalled hossam-0.5.3


In [2]:
from hossam import *
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from datetime import datetime as dt
from keras_tuner import Hyperband
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
# 분류 문제 유형에 맞는 성능평가 지표
from tensorflow.keras.metrics import AUC, F1Score
# 로지스틱 성능평가 함수
from sklearn.metrics import confusion_matrix
from tqdm.keras import TqdmCallback

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.4
현재 설치된 'hossam' 패키지 버전: 0.5.4

✅ 시각화를 위한 한글 글꼴(NotoSansKR-Regular)이 자동 적용되었습니다.


ModuleNotFoundError: No module named 'keras_tuner'

### 📝 [2] 데이터 가져오기

## 📘 #02. 탐색적 데이터 분석

### 📝 [1] 데이터 품질 검사

In [ ]:
origin = load_data("dogs")
origin.head()

In [ ]:
desc = origin.describe().T
# 숫자형 컬럼 이름만 추출
num_cols = origin.select_dtypes(include=np.number).columns
# 왜도 확인 및 로그 변환 필요성
for column in num_cols:
    skewness = origin[column].skew()
    if abs(skewness) < 0.5:
        strength = "week"
        log_transform = "not needed"
    elif abs(skewness) < 1:
        strength = "normal"
        log_transform = "recommended"
    else:
        strength = "strong"
        log_transform = "needed"
    desc.loc[column, "skewness"] = skewness
    desc.loc[column, "skewness_strength"] = strength
    desc.loc[column, "log_transform"] = log_transform
desc

In [ ]:
origin['dog'].value_counts()

## 📘 #03. 데이터 전처리

### 📝 [1] 종속변수 라벨링

분류를 수행할 경우 종속변수가 라벨링이 되어 있어야 하며, 데이터 타입이 정수 형태로 설정되어야 한다.

### 📝 [2] 훈련, 검증 데이터 분리

In [ ]:
df = origin.copy()
df['dog'] = df['dog'].map({'d': 0, 'j': 1})
df['dog'].value_counts()

In [ ]:
yname = "dog"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

## 📘 #03. 훈련 모델 적합

### 📝 [1] 하이퍼파라미터 튜닝 함수

| 구분 | 모델 | 활성화 함수 | 옵티마이저 | 손실함수 | 평가지표 | 과적합 판정 지표 | 대표예제 |
|------|------|-------------|------------|----------|----------|------------------|----------|
| 분류 | 이항분류 | relu → sigmoid | Adam | binary_crossentropy | accuracy, AUC, F1 | ValLoss − Train Loss, Train Accuracy − Val Accuracy | 타이타닉 |

In [ ]:
_, cols = x_train.shape
cols

In [ ]:
def tf_build(hp) -> Sequential:
    model = Sequential()
    # 입력층 정의
    model.add(Input(shape=(cols,)))
    # 은닉층 --> 유닛 수를 하이퍼파라미터로 조정
    model.add(
        Dense(
            units=hp.Choice("units", values=[4, 8, 16, 32, 64]),
            activation="relu",
        )
    )
    # -> 출력층: 1개의 뉴런 --> 하나의 값을 출력
    # -> 로지스틱 모델이기 때문에 활성화 함수는 sigmoid
    model.add(Dense(1, activation="sigmoid"))
    # 모델 학습 설정 (컴파일 단계)
    model.compile(
        optimizer="adam",
        # 이항분류이므로 손실률을 binary_crossentropy로 설정
        loss="binary_crossentropy",
        # accuracy: 정확도, AUC, F1Score
        metrics=["accuracy", AUC(name="auc"), F1Score(name="f1_score")],
    )
    return model

### 📝 [2] 튜너 객체 생성

### 📝 [3] 하이퍼파라미터 튜닝 수행

In [ ]:
tuner = Hyperband(
    hypermodel=tf_build,
    objective="accuracy",
    max_epochs=10,
    factor=3,
    seed=52,
    directory="tensor-tuning",
    project_name="tf_hyperband_%s" % dt.now().strftime("%Y%m%d%H%M%S"),
)
tuner

In [ ]:
%%time
tuner.search(
    x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test)
)
# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters()
if not best_hps:
    raise ValueError("No best hyperparameters found.")
print(f"""best hyperparameters: {best_hps[0].values}""")

### 📝 [4] 최종 모형 도출

콜백 설정에서 monitor 의 성능평가 지표를 정확도 accuracy 로 수정

## 📘 #05 성능평가

### 📝 [1] 성능평가 지표

In [ ]:
model = tuner.hypermodel.build(best_hps[0])
result = model.fit(
    x_train,
    y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        EarlyStopping(monitor='accuracy', patience=5, min_delta=0.001),
        ReduceLROnPlateau(monitor='accuracy', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

In [ ]:
# Train 성능 평가
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

# DataFrame 생성
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
# Gap 계산
for metric in ["loss", "accuracy", "auc", "f1_score"]:
    final_results[f"{metric}_gap"] = None
    final_results.loc[1, f"{metric}_gap"] = (
        final_results.loc[1, metric] - final_results.loc[0, metric]
    )
final_results

### 📝 [2] 학습 과정 확인

### 📝 [3] 학습 결과 시각화

| 그래프 | 목적 |
|--------|------|
| Loss curve | 과적합 확인 |
| Acc curve | 분류 성능 |

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1600 / 100, 600 / 100)
fig, ax = plt.subplots(1, 2, figsize=figsize, dpi=100)
fig.subplots_adjust(wspace=0.2, hspace=0.2)
# 훈련데이터의 손실률
sb.lineplot(
    data=history_df,
    x="epoch",
    y="loss",
    ax=ax[0],
    label="Train Loss"
)
# 검증 데이터의 손실률
sb.lineplot(
    data=history_df,
    x="epoch",
    y="val_loss",
    ax=ax[0],
    label="Validation Loss"
)
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Training vs Validation Loss")
ax[0].grid(True, alpha=0.3)
# 훈련데이터의 정확도
sb.lineplot(
    data=history_df,
    x="epoch",
    y="accuracy",
    ax=ax[1],
    label="Train Accuracy"
)
# 검증 데이터의 정확도
sb.lineplot(
    data=history_df,
    x="epoch",
    y="val_accuracy",
    ax=ax[1],
    label="Validation Accuracy"
)
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_title("Training vs Validation Accuracy")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

### 📝 [4] 분류결과 확인

#### ✏ 분류값 얻기

#### ✏ 분류값 판정

In [ ]:
pred = model.predict(x_test, verbose=0)
pred[:5]

In [ ]:
kdf = DataFrame({
    '실제값': y_test,
    '분류값': pred.flatten(),
})
kdf['분류결과'] = np.where(kdf['분류값'] >= 0.5, 1, 0)
kdf.head()

#### ✏ 혼동행렬 시각화

In [ ]:
cm = confusion_matrix(
    kdf['실제값'],
    kdf['분류결과']
)
figsize = (400 / 100, 300 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.heatmap(data=cm, annot=True, fmt="0.1f", linewidth=0.5)
ax.set_xlabel("")
ax.set_ylabel("")
ax.xaxis.tick_top()
plt.tight_layout()
plt.show()
plt.close()